# Pipeline Sequencial — Modelo 7 → Modelo 5

## Task 1 — Contrato de integração

Este notebook implementará futuramente o pipeline sequencial formado por:

1. **Modelo 7 — CatBoost:** previsão de risco por sistema PSD, no nível de estação e plataforma;
2. **Modelo 5 — Random Forest:** previsão de risco por porta individual dentro das plataformas selecionadas.

Nesta primeira task ainda não há execução dos modelos. O objetivo é registrar o contrato que deverá ser respeitado nas próximas etapas.

## 1. Fluxo acordado

```text
Dados disponíveis até o final do dia D
                    ↓
Modelo 7 — CatBoost por plataforma
                    ↓
Plataformas com probabilidade >= 0,317380
                    ↓
Modelo 5 — Random Forest por porta
                    ↓
Portas priorizadas para inspeção
```

O Modelo 7 é executado primeiro por trabalhar no nível mais amplo e indicar que alguma porta da plataforma poderá falhar. O Modelo 5 recebe somente as portas pertencentes às plataformas alertadas e tenta localizar qual porta deve ser priorizada. A diferença de granularidade é proposital: o CatBoost ajuda o Random Forest reduzindo o espaço de busca. O score do CatBoost não será usado como feature do Random Forest nesta primeira versão, porque o Modelo 5 não foi treinado com essa variável.

## 2. Comparação das interfaces atuais

| Item | Modelo 7 — CatBoost | Modelo 5 — Random Forest | Situação |
|---|---|---|---|
| Granularidade | Plataforma × dia | Porta × dia | Compatível, complementar e proposital |
| Momento da previsão | Final de `D`, incluindo eventos de `D` | Início de `D`, usando eventos até `D-1` | Precisa alinhar |
| Horizonte | `D+1` a `D+7` | `(D, D+7]` | Mesmo tamanho; referência precisa alinhar |
| Falha positiva | ZM01 com `falha_flag == True` | Qualquer ZM01 | Precisa corrigir |
| Plataforma | `id_psd = BUT_P1` | `id_plat = BUT_1` | Precisa converter |
| Porta | Não é a unidade prevista | `id_porta = BUT_1_05` | Precisa padronizar |
| Cobertura | 11 estações e 22 plataformas | 10 estações e 480 portas | `FRE` sem detalhamento por porta |
| Features | 21 registradas em metadata | 71 mantidas por SHAP | Congelar lista do Modelo 5 |
| Limiar | `0,317380` | `0,5183` | Manter separados |
| Artefato | `.cbm` e metadata disponíveis | Modelo não serializado | Precisa exportar |

## 3. Decisões do contrato

### Data e target comuns

- O pipeline será iniciado **ao final do dia `D`**.
- O Modelo 7 manterá as features disponíveis até o final de `D`.
- O Modelo 5 manterá inicialmente sua regra original, usando o histórico até `D-1`.
- Os dois modelos observarão o horizonte futuro de `D+1` a `D+7`.
- O Modelo 7 manterá o target de falha confirmada; o Modelo 5 manterá inicialmente o target de qualquer ZM01 da porta.
- O alinhamento completo de tempo e target será um aprimoramento posterior, não um bloqueio do primeiro pipeline.

### Chaves canônicas

```text
estacao = BUT
plataforma = 1
id_psd = BUT_P1
id_porta = BUT_P1_PDM05
```

As chaves internas dos modelos serão convertidas para esse padrão na fronteira do pipeline. Uma porta só poderá chegar ao Modelo 5 quando pertencer à mesma estação e plataforma alertadas pelo Modelo 7.

### Cobertura

Enquanto o Modelo 5 não incluir `FRE`, um alerta do Modelo 7 nessa estação deverá permanecer como alerta de plataforma sem detalhamento por porta. O alerta não poderá ser descartado silenciosamente.

## 4. Resultado da Task 1

Os modelos são conceitualmente complementares e a diferença de granularidade é o mecanismo central do pipeline: o Modelo 7 encontra a plataforma em risco e o Modelo 5 procura a porta dentro dela. Para automatizar a primeira versão, será necessário:

1. salvar a lista ordenada das 71 features;
2. serializar o Random Forest e registrar seu limiar e metadata;
3. implementar a conversão das chaves;
4. tratar plataformas sem cobertura no Modelo 5;
5. preservar a regra temporal original de cada modelo durante a inferência.

Em uma iteração posterior, o Modelo 5 poderá ser retreinado com o mesmo instante e o mesmo target do Modelo 7 para permitir uma avaliação unificada de ponta a ponta. Isso não impede o primeiro teste do encadeamento.

A especificação completa está em `docs/src/content/docs/sprints/Sprint 3/pipeline-sequencial-modelos.mdx`.